# NB_Bronze_To_Silver
Enterprise Banking Fabric

Author: Rakesh Soma

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from datetime import datetime

spark=SparkSession.builder.appName("EnterpriseBankingFabric-Silver").getOrCreate()

tables=[
"CountryMaster","CurrencyMaster","BranchMaster","CustomerTypeMaster",
"RiskCategoryMaster","OccupationMaster","AccountTypeMaster",
"LoanTypeMaster","CardTypeMaster","TransactionTypeMaster"
]

print("Silver Load Started:",datetime.now())

for table in tables:
    print(f"\nProcessing {table}")
    bronze=spark.table(f"Bronze.{table}")

    # Enterprise cleansing
    df=(bronze
        .dropDuplicates()
        .select([trim(col(c)).alias(c) if t=="string" else col(c)
                 for c,t in bronze.dtypes])
        .fillna("Unknown"))

    # Audit columns
    df=(df.withColumn("LoadDate",current_timestamp())
          .withColumn("LoadBy",lit("NB_Bronze_To_Silver"))
          .withColumn("RecordStatus",lit("Active")))

    # Example CDC hash
    business_cols=[c for c in df.columns if c not in ["LoadDate","LoadBy","RecordStatus"]]
    df=df.withColumn("RowHash",sha2(concat_ws("||",*map(col,business_cols)),256))

    # Example SCD Type-2 pattern (pseudo implementation)
    # Existing Silver table should contain:
    # EffectiveFrom, EffectiveTo, IsCurrent
    # Compare RowHash with current records to detect changes.
    #
    # changed = source.join(target.filter("IsCurrent=1"), key,"left")...
    #
    # Expire old rows:
    # update EffectiveTo=current_timestamp(), IsCurrent=0
    #
    # Insert new rows:
    df=(df.withColumn("EffectiveFrom",current_timestamp())
          .withColumn("EffectiveTo",lit(None).cast("timestamp"))
          .withColumn("IsCurrent",lit(1)))

    # Delta optimization settings
    spark.conf.set("spark.databricks.delta.optimizeWrite.enabled","true")
    spark.conf.set("spark.databricks.delta.autoCompact.enabled","true")

    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema","true")
       .saveAsTable(f"Silver.{table}"))

    print(f"{table} -> Silver loaded ({df.count()} rows)")

print("Silver Load Completed:",datetime.now())

# Recommended post-load commands:
# OPTIMIZE Silver.<table>;
# VACUUM Silver.<table> RETAIN 168 HOURS;
# ANALYZE TABLE Silver.<table> COMPUTE STATISTICS;
